# Full SEN2SR → Super-resolution → Spectral Index Comparison

This notebook reads the Copernicus ZIP from Google Drive, extracts it automatically, identifies the Sentinel-2 bands, prepares a 10-band stack, runs pretrained Full SEN2SR, and displays RGB, NDVI, NDWI and NDBI comparisons.

## Bands required by this notebook

The 10-band input order is:

`B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12`

Use Sentinel-2 L2A analytical data, not a JPEG/PNG visual export. Start with a small study-area download. The model prediction is not ground truth, and this notebook does not fine-tune the model or implement a spectral-index loss.

## Cell 1 — Install packages

Run this cell first in Google Colab.

In [1]:
%pip install -q sen2sr mlstac rasterio matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 2 — Imports and GPU check

In [2]:
import os, re, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from affine import Affine
import torch
import mlstac

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. In Colab, choose Runtime → Change runtime type → T4 GPU if available.")

Device: cuda
GPU: Tesla T4


## Cell 3 — Connect Drive and extract `Browser_images.zip`

Keep the ZIP in `My Drive/Datasets/`. This cell extracts it automatically into Colab's temporary `/content` storage. You do not need to unzip it manually.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

# Change this only if you put the ZIP in a different folder or used a different filename.
DRIVE_DATASET_DIR = "/content/drive/MyDrive/Project_Folder/Datasets"
ZIP_PATH = os.path.join(DRIVE_DATASET_DIR, "Browser_images.zip")

# Google Drive sometimes displays the name without the .zip extension.
if not os.path.exists(ZIP_PATH):
    alternative = os.path.join(DRIVE_DATASET_DIR, "Browser_images")
    if os.path.exists(alternative):
        ZIP_PATH = alternative

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(
        f"ZIP not found at: {ZIP_PATH}\n"
        "Check that Browser_images.zip is directly inside My Drive/Datasets."
    )

EXTRACT_DIR = "/content/copernicus_data"
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError(f"The selected file is not a valid ZIP archive: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

print("ZIP extracted from:", ZIP_PATH)
print("Extracted to:", EXTRACT_DIR)
print("\nRaster files found:")
raster_files = []
for root, dirs, filenames in os.walk(EXTRACT_DIR):
    for filename in filenames:
        if filename.lower().endswith((".tif", ".tiff", ".jp2")):
            path = os.path.join(root, filename)
            raster_files.append(path)
            print(path)

if not raster_files:
    raise FileNotFoundError(
        "No .tif, .tiff, or .jp2 files found inside the ZIP. "
        "The archive may contain a different format or a nested download that needs inspection."
    )

Mounted at /content/drive
ZIP extracted from: /content/drive/MyDrive/Project_Folder/Datasets/Browser_images.zip
Extracted to: /content/copernicus_data

Raster files found:
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B8A_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B04_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_True_color.tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B03_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B12_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B06_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B08_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B07_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B11_(Raw).tiff
/content/copernicus_data/2

## Cell 4 — Identify bands and build a 10-band GeoTIFF

This cell supports either:
- a single 10-band TIFF already ordered correctly, or
- separate band files named with standard tokens such as `..._B02_10m.jp2`.

If several files for the same band/resolution are present, the notebook stops rather than guessing. All bands must come from the same acquisition/granule. It aligns individual bands to the B02 grid.

In [4]:
BAND_NAMES = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]

# Case A: one multiband TIFF containing all ten bands.
multiband_candidates = []
for path in raster_files:
    if path.lower().endswith((".tif", ".tiff")):
        try:
            with rasterio.open(path) as src:
                if src.count == 10:
                    multiband_candidates.append(path)
        except Exception:
            pass

if len(multiband_candidates) == 1:
    INPUT_TIF = multiband_candidates[0]
    print("Using 10-band TIFF:", INPUT_TIF)
elif len(multiband_candidates) > 1:
    raise ValueError(
        "Found multiple 10-band TIFFs. Please keep only one intended input TIFF in the ZIP "
        "or set INPUT_TIF manually to the correct file."
    )
else:
    # Case B: separate band files. Match band names from standard SAFE/browser filenames.
    found = {}
    duplicates = {}
    for path in raster_files:
        upper = os.path.basename(path).upper()
        matches = [b for b in BAND_NAMES if re.search(rf"(^|[_\.-]){b}([_\.-]|$)", upper)]
        if not matches:
            continue
        b = matches[0]
        if b in found:
            duplicates.setdefault(b, [found[b]]).append(path)
        else:
            found[b] = path

    missing = [b for b in BAND_NAMES if b not in found]
    if duplicates:
        raise ValueError(
            "Multiple files found for these bands: " + str(duplicates) +
            ". The notebook will not guess which resolution/granule to use. "
            "Use one file per band from the same granule, or create a correctly ordered 10-band TIFF."
        )
    if missing:
        print("Band files identified:", found)
        raise ValueError(
            "Could not identify all ten required bands. Missing: " + ", ".join(missing) +
            ". The ZIP may contain a visualized export or separate TIFFs whose filenames do not identify raw band names. "
            "See the printed raster file list and we can adapt the loader."
        )

    # B02 is the reference 10 m grid.
    with rasterio.open(found["B02"]) as ref:
        ref_profile = ref.profile.copy()
        ref_transform, ref_crs = ref.transform, ref.crs
        height, width = ref.height, ref.width

    if ref_crs is None:
        raise ValueError("B02 file has no CRS. Use georeferenced raw Copernicus band files.")

    stack = []
    for b in BAND_NAMES:
        with rasterio.open(found[b]) as src:
            if src.crs is None:
                raise ValueError(f"{b} has no CRS metadata.")
            dest = np.zeros((height, width), dtype=np.float32)
            reproject(
                source=rasterio.band(src, 1),
                destination=dest,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=Resampling.bilinear
            )
            stack.append(dest)

    stack = np.stack(stack).astype(np.float32)
    INPUT_TIF = "/content/sentinel2_10band_stack.tif"
    profile = ref_profile.copy()
    profile.update(driver="GTiff", count=10, height=height, width=width,
                   dtype="float32", compress="deflate", nodata=None)
    with rasterio.open(INPUT_TIF, "w", **profile) as dst:
        dst.write(stack)
    print("Created stack:", INPUT_TIF)

print("\nInput TIFF:", INPUT_TIF)
with rasterio.open(INPUT_TIF) as src:
    print("Bands:", src.count, "| shape:", (src.height, src.width), "| dtype:", src.dtypes)
    print("CRS:", src.crs, "| resolution:", src.res)

Created stack: /content/sentinel2_10band_stack.tif

Input TIFF: /content/sentinel2_10band_stack.tif
Bands: 10 | shape: (97, 95) | dtype: ('float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32')
CRS: EPSG:4326 | resolution: (8.967368421044859e-05, 7.736082474227673e-05)


## Cell 5 — Load input and set reflectance scaling

Copernicus Browser analytical exports may use different numeric scaling than original SAFE product values. Inspect the printed range. For common Sentinel-2 SAFE integer surface-reflectance data, values are often scaled by 10000; a browser export may instead be scaled to its selected output bit depth.

**Do not blindly divide by 10000 if the values are already scaled differently.** This cell uses a simple range-based initial guess, but verify it before scientific use.

In [5]:
with rasterio.open(INPUT_TIF) as src:
    raw = src.read().astype(np.float32)
    input_profile = src.profile.copy()
    input_transform = src.transform
    input_crs = src.crs
    nodata = src.nodata

if raw.shape[0] != 10:
    raise ValueError(f"Expected 10 bands, got {raw.shape[0]}.")

finite = raw[np.isfinite(raw)]
print("Raw pixel range:", float(np.nanpercentile(finite, 1)), "to", float(np.nanpercentile(finite, 99)))
print("dtype:", raw.dtype, "| nodata:", nodata)

# EDIT THIS after inspecting the range above:
# - use 10000.0 for common SAFE integer reflectance values scaled by 10000
# - use 1.0 if already stored as reflectance in the 0–1 range
# - for Copernicus Browser 16-bit rendered analytical exports, inspect the export scaling/documentation first.
SCALE_FACTOR = 10000.0

X_np = raw / SCALE_FACTOR
if nodata is not None:
    X_np[raw == nodata] = np.nan
print("Scaled reflectance range (1st–99th percentile):",
      float(np.nanpercentile(X_np, 1)), "to", float(np.nanpercentile(X_np, 99)))

# Fill invalid pixels only to prevent model crashes; cloud/NoData masking is still needed for valid science.
X_np = np.nan_to_num(X_np, nan=0.0, posinf=0.0, neginf=0.0)
X = torch.from_numpy(X_np).float().to(device)
print("Model input shape:", tuple(X.shape))

Raw pixel range: 1890.429931640625 to 19916.0
dtype: float32 | nodata: None
Scaled reflectance range (1st–99th percentile): 0.18904300034046173 to 1.9916000366210938
Model input shape: (10, 97, 95)


## Cell 6 — Download and load pretrained **full SEN2SR**

This loads the full 10-band SEN2SR model entry from the official model repository, not the Lite entry. This notebook performs pretrained inference only; it does not fine-tune the model.

In [6]:
import os
import mlstac

MODEL_DIR = "/content/model/SEN2SR_FULL"
os.makedirs(MODEL_DIR, exist_ok=True)

mlstac.download(
    file="https://huggingface.co/tacofoundation/SEN2SR/resolve/main/SEN2SR/main/mlm.json",
    output_dir=MODEL_DIR,
)

print("Full SEN2SR model files downloaded to:", MODEL_DIR)
print("Files:", os.listdir(MODEL_DIR))

Full SEN2SR model files downloaded to: /content/model/SEN2SR_FULL
Files: ['sr_hard_constraint.safetensor', 'hard_constraint.safetensor', 'mlm.json', 'f2_hard_constraint.safetensor', 'model.safetensor', 'sr_model.safetensor', 'f2_model.safetensor', 'load.py', 'example_data.safetensor']


## Cell 7 — Run tiled full-model inference

Tiled inference keeps memory use manageable and avoids assuming the full image can be passed through the model in one call. The code checks the model's output shape instead of silently assuming the scale.

In [8]:
%pip install -q ninja packaging

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 21.4 MB/s eta 0:00:00


In [9]:
%pip install mamba-ssm --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 1.0 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.9/247.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

from mamba_ssm.ops.selective_scan_interface import selective_scan_fn

print("Mamba import successful!")

PyTorch: 2.14.1+cu130
CUDA: 13.0
GPU available: True


ImportError: /usr/local/lib/python3.11/dist-packages/selective_scan_cuda.cpython-311-x86_64-linux-gnu.so: undefined symbol: _ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_ib

In [ ]:
# Tile settings. The official full model may have strict input patch constraints.
PATCH = 128
OVERLAP = 32
STRIDE = PATCH - OVERLAP

if X.ndim != 3 or X.shape[0] != 10:
    raise ValueError(f"Expected X with shape (10, H, W), got {tuple(X.shape)}")

_, H, W = X.shape
pad_h = max(0, PATCH - H)
pad_w = max(0, PATCH - W)
X_pad = torch.nn.functional.pad(X, (0, pad_w, 0, pad_h), mode="replicate")
_, Hp, Wp = X_pad.shape

# Probe one patch to learn the model's actual output scale and band count.
with torch.inference_mode():
    probe = model(X_pad[:, :PATCH, :PATCH].unsqueeze(0))
if probe.ndim != 4:
    raise ValueError(f"Expected model output (N,C,H,W); got {tuple(probe.shape)}")

out_bands = probe.shape[1]
tile_h, tile_w = probe.shape[-2:]
if tile_h % PATCH or tile_w % PATCH:
    raise ValueError(
        f"Unexpected tile output size {tile_h}x{tile_w} for {PATCH}x{PATCH} input."
    )
scale_y, scale_x = tile_h // PATCH, tile_w // PATCH
print("Model patch output:", tuple(probe.shape))
print(f"Detected scale: {scale_x}x horizontally, {scale_y}x vertically")

out = torch.zeros(
    (out_bands, Hp * scale_y, Wp * scale_x),
    device=device, dtype=torch.float32
)
weights = torch.zeros(
    (1, Hp * scale_y, Wp * scale_x),
    device=device, dtype=torch.float32
)

with torch.inference_mode():
    for y in range(0, Hp - PATCH + 1, STRIDE):
        for x in range(0, Wp - PATCH + 1, STRIDE):
            # Ensure final tiles reach the lower/right edge.
            y0 = min(y, Hp - PATCH)
            x0 = min(x, Wp - PATCH)
            patch = X_pad[:, y0:y0+PATCH, x0:x0+PATCH].unsqueeze(0)
            pred = model(patch).squeeze(0).float()

            if pred.shape[0] != out_bands:
                raise ValueError("Model output channel count changed between tiles.")
            oy, ox = y0 * scale_y, x0 * scale_x
            ph, pw = pred.shape[-2:]
            out[:, oy:oy+ph, ox:ox+pw] += pred
            weights[:, oy:oy+ph, ox:ox+pw] += 1

superX = out / weights.clamp_min(1)
superX = superX[:, :H*scale_y, :W*scale_x]

print("Input:", tuple(X.shape))
print("Output:", tuple(superX.shape))
if superX.ndim != 3 or superX.shape[0] != 10:
    raise ValueError(
        f"Expected a 10-band output, got {tuple(superX.shape)}. "
        "Check the full model's input/output contract."
    )
print("Tiled inference complete.")

## Cell 8 — Calculate NDVI, NDWI and NDBI

- NDVI = (B08 − B04) / (B08 + B04)
- NDWI (Green–NIR) = (B03 − B08) / (B03 + B08)
- NDBI = (B11 − B08) / (B11 + B08)

These indices are only as meaningful as the band alignment, scaling, masking and output-band contract.

In [ ]:
BAND = {name: i for i, name in enumerate(BAND_NAMES)}

def normalized_difference(a, b, eps=1e-6):
    denom = a + b
    out = np.full(denom.shape, np.nan, dtype=np.float32)
    valid = np.isfinite(a) & np.isfinite(b) & (np.abs(denom) > eps)
    out[valid] = (a[valid] - b[valid]) / denom[valid]
    return out

def calc_indices(arr):
    return {
        "NDVI": normalized_difference(arr[BAND["B08"]], arr[BAND["B04"]]),
        "NDWI": normalized_difference(arr[BAND["B03"]], arr[BAND["B08"]]),
        "NDBI": normalized_difference(arr[BAND["B11"]], arr[BAND["B08"]]),
    }

input_np = X.detach().cpu().numpy()
output_np = superX.detach().cpu().numpy()
input_indices = calc_indices(input_np)
output_indices = calc_indices(output_np)
print("Indices computed.")

## Cell 9 — Display original RGB, predicted RGB, and all index maps

In [ ]:
def rgb_display(arr):
    rgb = np.stack([arr[BAND["B04"]], arr[BAND["B03"]], arr[BAND["B02"]]], axis=-1)
    lo, hi = np.nanpercentile(rgb, [2, 98])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = 0.0, 1.0
    return np.clip((rgb - lo) / (hi - lo), 0, 1)

fig, axes = plt.subplots(4, 2, figsize=(14, 22))
axes[0,0].imshow(rgb_display(input_np)); axes[0,0].set_title("Input RGB")
axes[0,1].imshow(rgb_display(output_np)); axes[0,1].set_title("SEN2SR output RGB")

for row, name in enumerate(["NDVI", "NDWI", "NDBI"], start=1):
    cmap = "RdYlGn" if name == "NDVI" else "RdBu"
    a = axes[row,0].imshow(input_indices[name], cmap=cmap, vmin=-1, vmax=1)
    axes[row,0].set_title("Input " + name)
    b = axes[row,1].imshow(output_indices[name], cmap=cmap, vmin=-1, vmax=1)
    axes[row,1].set_title("SEN2SR output " + name)
    fig.colorbar(a, ax=axes[row,0], fraction=0.046, pad=0.04)
    fig.colorbar(b, ax=axes[row,1], fraction=0.046, pad=0.04)

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Cell 10 — Difference maps and diagnostic statistics

In [ ]:
def nearest_resize(arr, h, w):
    yi = np.minimum((np.arange(h) * arr.shape[0] / h).astype(int), arr.shape[0]-1)
    xi = np.minimum((np.arange(w) * arr.shape[1] / w).astype(int), arr.shape[1]-1)
    return arr[np.ix_(yi, xi)]

fig, axes = plt.subplots(3, 2, figsize=(14, 15))
print("Index | MAE | RMSE | valid pixels")
for i, name in enumerate(["NDVI", "NDWI", "NDBI"]):
    inp = nearest_resize(input_indices[name], *output_indices[name].shape)
    diff = output_indices[name] - inp
    im = axes[i,0].imshow(diff, cmap="RdBu", vmin=-0.5, vmax=0.5)
    axes[i,0].set_title(name + " difference (output - resized input)")
    fig.colorbar(im, ax=axes[i,0], fraction=0.046, pad=0.04)
    vals = diff[np.isfinite(diff)]
    axes[i,1].hist(vals, bins=60)
    axes[i,1].set_title(name + " difference distribution")
    axes[i,1].set_xlabel("Index difference")
    mae = float(np.mean(np.abs(vals))) if vals.size else float("nan")
    rmse = float(np.sqrt(np.mean(vals**2))) if vals.size else float("nan")
    print(f"{name:5s} | {mae:.6f} | {rmse:.6f} | {vals.size}")
plt.tight_layout()
plt.show()

## Cell 11 — Save model output to Drive as a 10-band GeoTIFF

In [ ]:
from affine import Affine

OUTPUT_TIF = "/content/drive/MyDrive/Datasets/sen2sr_output.tif"
profile = input_profile.copy()
profile.update(
    driver="GTiff", count=10,
    height=output_np.shape[1], width=output_np.shape[2],
    dtype="float32", nodata=None, compress="deflate",
    transform=input_transform * Affine.scale(1/scale_x, 1/scale_y)
)
with rasterio.open(OUTPUT_TIF, "w", **profile) as dst:
    dst.write(output_np.astype(np.float32))

print("Saved output to Google Drive:", OUTPUT_TIF)

## Important notes

- If the ZIP contains only visualized layers or fewer than ten raw bands, Cell 4 will stop. Do not force the model to run on RGB-only data.
- This notebook now requests the **full SEN2SR** model entry, not `SEN2SRLite`. If the official model URL, dependency versions, or checkpoint contract has changed, model loading may fail; inspect the full error rather than silently switching models.
- The ten-band order is assumed to be `B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12`. Confirm the input files and reflectance scaling before interpreting results.
- The code uses 128×128 input tiles with 32-pixel overlap and averages overlapping predictions. If the full model requires a different patch size or returns a different scale, the probe/checks will show that.
- Difference maps compare output indices with a resized input index. This is a diagnostic visualization, **not ground truth**.
- This notebook performs pretrained inference only. It does not fine-tune the model with a Multi-Index Spectral Loss.
- The separate-band stacker resamples bands to the B02 grid. Because Sentinel-2 bands have different native resolutions, this preprocessing choice affects the scientific interpretation of the output.